In [ ]:
import os
os.environ["TORCH_DYNAMO_DISABLE"] = "1"                          
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"                                             

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
import multiprocessing


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# =================================================================
                                
# =================================================================
class FourierFeatureMapping(nn.Module):
    def __init__(self, in_dims=2, mapping_size=128, scale=10.0):
        super().__init__()
                                  
        self.register_buffer('B', torch.randn(in_dims, mapping_size) * scale)

    def forward(self, x):
        # x: (Batch, 2)
        # x @ B: (Batch, mapping_size)
        x_proj = (2.0 * np.pi * x) @ self.B
                                          
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)

# =================================================================
               
# =================================================================
class DenseResBlock(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.fc1 = nn.Linear(size, size)
        self.fc2 = nn.Linear(size, size)
        self.norm1 = nn.LayerNorm(size)
        self.norm2 = nn.LayerNorm(size)

    def forward(self, x):
        res = x
        x = self.norm1(x)
        x = F.silu(self.fc1(x))
        x = self.norm2(x)
        x = F.silu(self.fc2(x))
        return x + res

# =================================================================
                       
# =================================================================
class HighFidelityNeuralVF(nn.Module):
    def __init__(self, n_poles=8, width=256, fourier_size=128):
        super().__init__()
        self.n_poles = n_poles
        
                                         
        self.fourier_map = FourierFeatureMapping(in_dims=2, mapping_size=fourier_size, scale=5.0)
        
                                 
        in_features = fourier_size * 2 + 2 
        self.in_layer = nn.Linear(in_features, width)
        
                           
        self.res_layers = nn.ModuleList([DenseResBlock(width) for _ in range(6)])
        
                
        self.out_layer = nn.Linear(width, 2 * n_poles)

    def forward(self, x):
        # x shape: (Batch, 2)
        
                
        x_fourier = self.fourier_map(x)
                                            
        x_concat = torch.cat([x, x_fourier], dim=-1)
        
        h = F.silu(self.in_layer(x_concat))
        for layer in self.res_layers:
            h = layer(h)
        out = self.out_layer(h)
        
                             
        log_tau_k = out[:, 0 : self.n_poles]
        log_w_k = out[:, self.n_poles :]
        
        tau_k = 10.0 ** log_tau_k
        w_k = 10.0 ** log_w_k
                             
        w_k_sum = torch.sum(w_k, dim=1, keepdim=True) + 1e-12        
        w_k = w_k / w_k_sum
        
        return tau_k, w_k, log_tau_k, log_w_k

In [ ]:
def physics_informed_loss( p_tau, p_w, p_log_tau, p_log_w, inputs, s_bench):

                                    
    log_t_cc = inputs[:, 0:1] 
    c_cc = inputs[:, 1:2]     
    t_cc = 10 ** log_t_cc     

    s_tensor = s_bench.view(1, -1) 
    y_true = 1.0 / (1.0 + (s_tensor * t_cc) ** c_cc)
    
    s_3d = s_bench.view(1, -1, 1)
    tau_3d = p_tau.unsqueeze(1)
    w_3d = p_w.unsqueeze(1)
    
    denom = 1.0 + s_3d * tau_3d
    terms = w_3d / denom        
    y_pred = torch.sum(terms, dim=-1)
    
    loss_phys = torch.mean(torch.abs((y_true - y_pred)/y_true)**2)
    
    return loss_phys

In [ ]:
import numpy as np
from scipy.optimize import least_squares
from tqdm import tqdm
from joblib import Parallel, delayed
import multiprocessing

def _fit_single_trf_worker(tau_cc, c_cc, n_poles):
    """单点 TRF 拟合 Worker"""
    omega = np.logspace(-2, 6, 100)
    s = 1j * omega
    Y_true = 1.0 / (1.0 + (s * tau_cc)**c_cc)
    Y_target = np.concatenate([np.real(Y_true), np.imag(Y_true)])
    W = np.concatenate([1.0/(np.abs(Y_true)+1e-6), 1.0/(np.abs(Y_true)+1e-6)])

    def residual(params):
        tk = params[:n_poles]
        wk = params[n_poles:-1]
        d0 = params[-1]
        pred = d0 + np.sum(wk[None, :] / (1.0 + s[:, None] * tk[None, :]), axis=1)
        return (np.concatenate([np.real(pred), np.imag(pred)]) - Y_target) * W

    p0 = np.concatenate([np.logspace(np.log10(tau_cc)-2, np.log10(tau_cc)+2, n_poles), 
                         np.ones(n_poles)/n_poles, [0.0]])
    bounds = (np.zeros(2*n_poles+1), np.inf)
    bounds[0][:n_poles] = 1e-9

    res = least_squares(residual, p0, bounds=bounds, method='trf', ftol=1e-9)
    
    tau_k, w_k, d0 = res.x[:n_poles], res.x[n_poles:-1], res.x[-1]
    idx = np.argsort(tau_k)
    return d0, tau_k[idx], w_k[idx]

def generate_trf_dataset(n_samples=2000, n_poles=8):
    """使用 Joblib 并行生成高精度数据集 (完美兼容 Jupyter/Windows)"""
    n_cores = 48
    print(f"[*] 正在使用 Joblib ({n_cores} 核) 生成 TRF 教师数据...")
    
    tau_samples = 10**np.random.uniform(-3,0, n_samples)
    c_samples = np.random.uniform(0, 1, n_samples)
    inputs = np.stack([np.log10(tau_samples), c_samples], axis=1)

    
          
    results_d0 = inputs[:,0]
    results_tau = inputs[:,0]
    results_w = inputs[:,0]
            
    return np.array(inputs), np.array(results_d0), np.array(results_tau), np.array(results_w)

                                   
X_np, Y_d0_np, Y_tau_np, Y_w_np = generate_trf_dataset(n_samples=10000, n_poles=12)
print("生成完毕！数据维度:", X_np.shape, Y_tau_np.shape)


In [ ]:
X_np.shape

In [ ]:
        
trf_train_data_dict = {
    'X_train': X_np,
    'Y_d0': Y_d0_np,
    'Y_tau': Y_tau_np,
    'Y_w': Y_w_np
}

np.save('trf_train_data_saved.npy', trf_train_data_dict)
print("[*] 数据已保存为 trf_train_data_saved.npy")

In [ ]:

# =================================================================
              
# =================================================================

N_POLES = 12
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_loaded = np.load('trf_train_data_saved.npy', allow_pickle=True).item()
X_np = data_loaded['X_train']
Y_d0_np = data_loaded['Y_d0']
Y_tau_np = data_loaded['Y_tau']
Y_w_np = data_loaded['Y_w']
print("[*] 已加载 trf_train_data_dict.npy 数据。")

# X_np, Y_d0_np, Y_tau_np, Y_w_np = generate_trf_dataset(n_samples=300, n_poles=N_POLES)

                   
dataset = TensorDataset(
    torch.tensor(X_np, dtype=torch.float32).to(device),
    torch.tensor(Y_d0_np, dtype=torch.float32).to(device),
    torch.tensor((Y_tau_np), dtype=torch.float32).to(device),
    torch.tensor(Y_w_np, dtype=torch.float32).to(device)
)
dataloader = DataLoader(dataset, batch_size=512, shuffle=True)



In [ ]:
              
model = HighFidelityNeuralVF(N_POLES).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=500, factor=0.5)

                                           
freqs_bench = torch.logspace(-2, 6, 100).to(device)
s_bench = 1j * 2 * torch.pi * freqs_bench


In [ ]:

         
epochs = 6000
print("\n[*] 开始 PyTorch 网络训练...")
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_x, batch_d0, batch_tau, batch_w in dataloader:
            optimizer.zero_grad()
            
                              
            p_tau, p_w, p_log_tau, p_log_w = model(batch_x)
            
                       
            loss = physics_informed_loss(
                 p_tau, p_w, p_log_tau, p_log_w, 
                batch_x,  s_bench
            )
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    scheduler.step(avg_loss)

    if (epoch + 1) % 100 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.6e} | LR: {optimizer.param_groups[0]['lr']:.2e}")

print("[*] 训练完成！")


In [ ]:
'''# 重新初始化模型结构，并加载权重
# 假设你的模型类名是 Net（请根据你的实际模型类名替换）
# model = Net()  # 如果未定义Net，则按你的实际模型类名
model.load_state_dict(torch.load("model_trained.pth"))
model.eval()
print("[*] 模型已从 model_trained.pth 加载并进入推理模式")'''


In [ ]:
'''with torch.no_grad():
    # 扩充10倍: 将X_np复制10次，进行10倍推断
    X_np_expanded = np.tile(X_np, (100, 1))  # (N*10, dim)
    with torch.no_grad():
        d0_pred, tau_pred, w_pred, _, _ = model(torch.tensor(X_np_expanded, dtype=torch.float32).to(device))
    '''

In [ ]:
# =================================================================
                     
# =================================================================
model.eval()
num= 10

test_tau_val = 10**X_np[num,0]
test_c_val = X_np[num,1]
test_tau_val = 0.01
test_c_val = 0.15


In [ ]:

print(f"\n[*] 正在验证测试点 (tau={test_tau_val}, c={test_c_val})...")

                                                
x_test = torch.tensor([[np.log10(test_tau_val), test_c_val]], dtype=torch.float32).to(device)

with torch.no_grad():
    tau_pred, w_pred, _, _ = model(x_test)
    
                        
tau_np = tau_pred.cpu().numpy()[0]
w_np = w_pred.cpu().numpy()[0]

      
freqs = np.logspace(-2, 6, 100)
s = 1j * 2 * np.pi * freqs

    
Y_true = 1.0 / (1.0 + (s * test_tau_val)**test_c_val)
     
Y_pred =np.sum(w_np[:, None] / (1.0 + s[None, :] * tau_np[:, None]), axis=0)


# Y_pred_2 = Y_d0_np[num] + np.sum(Y_w_np[num][:, None] / (1.0 + s[None, :] * Y_tau_np[num][:, None]), axis=0)


In [ ]:
result = np.sum(w_np)
print("sum(w_np) =", result)

In [ ]:

plt.figure(figsize=(12, 8))

                  
plt.subplot(2, 2, 1)
plt.semilogx(freqs, Y_true.real, 'k-', lw=3, label="True Cole-Cole (Real)")
plt.semilogx(freqs, Y_pred.real, 'r--', lw=2, label="PyTorch Neural-VF (Real)")
#plt.semilogx(freqs, Y_pred_2.real, 'b', lw=2, label="Fit (Real)")

plt.title(f"Real Part (tau={test_tau_val}, c={test_c_val})")
plt.legend(); plt.grid(True, alpha=0.3)

                  
plt.subplot(2, 2, 2)
plt.semilogx(freqs, Y_true.imag, 'k-', lw=3, label="True Cole-Cole (Imag)")
plt.semilogx(freqs, Y_pred.imag, 'r--', lw=2, label="PyTorch Neural-VF (Imag)")
#plt.semilogx(freqs, Y_pred_2.imag, 'b', lw=2, label="Fit (Imag)")
plt.title(f"Imag Part (tau={test_tau_val}, c={test_c_val})")
plt.legend(); plt.grid(True, alpha=0.3)

             
plt.subplot(2, 2, 3)
                 
err_nn_real = 100 * np.abs(Y_true.real - Y_pred.real) / (np.maximum(np.abs(Y_true.real), 1e-12))

plt.loglog(freqs, err_nn_real, 'r-', label="Real Part RelErr")
#plt.loglog(freqs, err_nn_real2, 'b-', label="Real2 Part RelErr")

plt.title("Relative Error (Real) (%)")
plt.axhline(1.0, color='k', ls=':', label='1% Threshold')
plt.legend(); plt.grid(True, which="both", alpha=0.3)

             
plt.subplot(2, 2, 4)
                 
err_nn_imag = 100 * np.abs(Y_true.imag - Y_pred.imag) / np.maximum(np.abs(Y_true.imag), 1e-12)

plt.loglog(freqs, err_nn_imag, 'r-', label="Imag Part RelErr")
#plt.loglog(freqs, err_nn_imag2, 'b-', label="Imag2 Part RelErr")

plt.title("Relative Error (Imag) (%)")
plt.axhline(1.0, color='k', ls=':', label='1% Threshold')
plt.legend(); plt.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
       
                  
torch.save(model.state_dict(), "model_trained.pth")
print("[*] 模型已保存为 model_trained.pth")

       
                 
                              
                                      
model.load_state_dict(torch.load("model_trained.pth"))
model.eval()
print("[*] 模型已从 model_trained.pth 加载并进入推理模式")